# Training transformers

The first cells pull the committed code from branch `BRANCH` and prepare Colab (HF packages, data, W&B).
Each following cell is a separate run: change the `Config(...)` and run it. Runs go to W&B `transformers-for-ner`;
probs.npz and the submission of every run are logged there as an artifact, for src.ensemble.

In [ ]:
import os
from getpass import getpass

os.environ["KAGGLE_API_TOKEN"] = getpass("KAGGLE_API_TOKEN: ")
os.environ["WANDB_API_KEY"] = getpass("WANDB_API_KEY: ")
print({k: len(os.environ[k]) for k in ("KAGGLE_API_TOKEN", "WANDB_API_KEY")})

In [ ]:
BRANCH = "main"
REPO = "https://github.com/mykhailoisupov/RNN_for_NER.git"
ROOT = "/content/RNN_for_NER"

import os, subprocess, sys

if not os.path.exists(ROOT):
    subprocess.run(["git", "clone", "-q", REPO, ROOT], check=True)
for cmd in (["fetch", "-q", "origin"], ["checkout", "-q", BRANCH], ["reset", "-q", "--hard", f"origin/{BRANCH}"]):
    subprocess.run(["git", "-C", ROOT, *cmd], check=True)
print(subprocess.run(["git", "-C", ROOT, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

os.chdir(ROOT)
sys.path.insert(0, ROOT)
for name in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[name]

from src.colab import setup
setup(transformer=True)

In [ ]:
from src.transformer.config import Config, DataConfig, ModelConfig, OptimConfig
from src.transformer.train import run

## Smoke test

A tiny random model, a small subset, W&B disabled: checks that everything runs (F1 stays 0).

In [ ]:
run(Config(model=ModelConfig(name="hf-internal-testing/tiny-xlm-roberta"),
           data=DataConfig(max_length=56, stride=16, train_size=300, val_size=200),
           epochs=1, wandb_mode="disabled", name="smoke"))

## XLM-RoBERTa base

In [ ]:
cfg = Config(
    model=ModelConfig(name="xlm-roberta-base"),
    data=DataConfig(train_size=50_000, max_length=512, batch_size=16),
    optim=OptimConfig(lr=3e-5),
    epochs=3,
)
run(cfg)

## mDeBERTa-v3 base

In [ ]:
cfg = Config(
    model=ModelConfig(name="microsoft/mdeberta-v3-base"),
    data=DataConfig(train_size=50_000, max_length=512, batch_size=16),
    optim=OptimConfig(lr=3e-5),
    epochs=3,
)
run(cfg)